# NLP Text Analysis Pipeline

### Extractive Summarization, Sentiment Analysis and Machine Translation

This project implements a compact Natural Language Processing (NLP) pipeline over a Spanish text discussing the benefits and risks of technology.

The workflow combines three tasks:

1. **Extractive summarization** using a deterministic sentence-selection strategy.
2. **Sentiment analysis** using a multilingual BERT model from Hugging Face.
3. **Spanish-to-English machine translation** using a pretrained MarianMT model.

The objective is to demonstrate how rule-based text processing and pretrained Transformer models can be combined in a single reproducible NLP workflow.

## 1. Setup and Input Text

The input is a Spanish text that discusses both the positive impact of technology and challenges such as the digital divide, privacy and data security.

The required libraries are imported below. The Hugging Face `transformers` library is used for sentiment analysis and machine translation.

In [ ]:
# Install dependencies if required
# !pip install transformers sentencepiece torch

import re

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

In [ ]:
text = """
La tecnología ha revolucionado prácticamente todos los aspectos de nuestras vidas, desde cómo nos
comunicamos hasta cómo realizamos nuestras tareas diarias, cambiando también la manera en que
interactuamos con el mundo que nos rodea. Su impacto es evidente en áreas tan diversas como la
educación, la salud, el transporte, y el comercio, mejorando la eficiencia, la accesibilidad y la
conectividad a niveles nunca antes imaginados. Sin embargo, a pesar de los innumerables beneficios que
nos brinda, la tecnología también trae consigo desafíos complejos y significativos que no podemos
ignorar.

Uno de los retos más destacados es la brecha digital, que exacerba las desigualdades sociales y
económicas entre quienes tienen acceso a las tecnologías y quienes no. Esta disparidad limita las
oportunidades educativas, laborales y sociales de muchas personas en diversas partes del mundo,
dejando atrás a comunidades enteras en el desarrollo tecnológico. Además, la expansión de la tecnología
ha planteado serias preocupaciones en torno a la privacidad y la seguridad de los datos personales, ya
que gran parte de nuestras actividades diarias, desde transacciones bancarias hasta interacciones
sociales, dejan un rastro digital que puede ser explotado.

Es crucial reflexionar detenidamente sobre estos aspectos para mitigar los riesgos asociados al avance
tecnológico y garantizar que su impacto sea positivo para la mayor cantidad de personas posible. Sólo a
través de un análisis consciente y acciones responsables podemos aspirar a construir un futuro más
equilibrado, donde la tecnología sea una herramienta al servicio de toda la humanidad, en lugar de
convertirse en una fuente de desigualdad o amenaza.
"""

## 2. Extractive Summarization

A lightweight extractive strategy is used to build a concise summary directly from sentences in the original text.

The text is first segmented into sentences using a regular expression. The summary then combines the opening sentence, a sentence introducing one of the main challenges, and the concluding sentence.

This approach is intentionally simple and deterministic. It does **not** rank sentence importance with a learned summarization model, but it produces a reproducible extractive summary without additional model dependencies.

In [ ]:
# Split the text into sentences
sentences = re.split(r'(?<=[.!?])\s+', text.strip())

# Deterministic extractive summary:
# opening context + main challenge + concluding idea
summary = (
    sentences[0] + " " +
    sentences[3] + " " +
    sentences[-1]
)

print("===== EXTRACTIVE SUMMARY =====")
print(summary)

===== EXTRACTIVE SUMMARY =====
La tecnología ha revolucionado prácticamente todos los aspectos de nuestras vidas, desde cómo nos 
comunicamos hasta cómo realizamos nuestras tareas diarias, cambiando también la manera en que 
interactuamos con el mundo que nos rodea. Uno de los retos más destacados es la brecha digital, que exacerba las desigualdades sociales y 
económicas entre quienes tienen acceso a las tecnologías y quienes no. Sólo a 
través de un análisis consciente y acciones responsables podemos aspirar a construir un futuro más 
equilibrado, donde la tecnología sea una herramienta al servicio de toda la humanidad, en lugar de 
convertirse en una fuente de desigualdad o amenaza.


### 2.1 Summarization Note

Pretrained Spanish summarization models were initially considered, but the original exercise encountered compatibility and checkpoint issues in the execution environment.

For this reason, the final implementation uses a deterministic extractive strategy. This keeps the pipeline stable and reproducible, while also making the limitation explicit: the summary is based on predefined sentence positions rather than semantic importance learned by a summarization model.

## 3. Sentiment Analysis

Sentiment is estimated with the pretrained multilingual model:

`nlptown/bert-base-multilingual-uncased-sentiment`

The model returns a rating from one to five stars. For this project, the original ratings are mapped to three broader categories:

- **1–2 stars:** negative
- **3 stars:** neutral
- **4–5 stars:** positive

To stay within the scope of the original exercise, sentiment is evaluated on the first 512 characters of the text. This is an important limitation because the full document contains both positive and critical arguments.

In [ ]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

raw_sentiment = sentiment_pipeline(text[:512])[0]
label = raw_sentiment["label"]

if label in ["4 stars", "5 stars"]:
    sentiment = "positive"
elif label in ["1 star", "2 stars"]:
    sentiment = "negative"
else:
    sentiment = "neutral"

print("===== SENTIMENT =====")
print(f"Detected sentiment: {sentiment}")
print(f"Raw model output: {raw_sentiment}")

===== SENTIMENT =====
Detected sentiment: positive
Raw model output: {'label': '5 stars', 'score': 0.4778934419155121}


### 3.1 Interpretation

The model classifies the analyzed portion of the text as **positive**, returning `5 stars` with a confidence score of approximately **0.478** in the original run.

This result should be interpreted cautiously. The source text is nuanced: it presents clear benefits of technology but also discusses the digital divide, privacy and data-security risks. In addition, the implementation evaluates only the first 512 characters rather than the complete text.

The experiment therefore illustrates both the usefulness and the limitations of applying a general-purpose pretrained sentiment model to long, mixed-polarity content.

## 4. Spanish-to-English Machine Translation

Machine translation is performed with the pretrained MarianMT model:

`Helsinki-NLP/opus-mt-es-en`

The text is split into paragraphs before translation to reduce the risk of truncation. Each paragraph is tokenized independently and translated using beam search with four beams.

In [ ]:
translation_model_name = "Helsinki-NLP/opus-mt-es-en"

translation_tokenizer = AutoTokenizer.from_pretrained(translation_model_name)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(translation_model_name)

# Split the document into paragraphs to reduce truncation risk
paragraphs = text.strip().split("\n\n")
translated_paragraphs = []

for paragraph in paragraphs:
    inputs = translation_tokenizer(
        paragraph,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    translated_tokens = translation_model.generate(
        **inputs,
        max_length=500,
        num_beams=4,
        early_stopping=True
    )

    translated_paragraph = translation_tokenizer.decode(
        translated_tokens[0],
        skip_special_tokens=True
    )

    translated_paragraphs.append(translated_paragraph)

translation = "\n\n".join(translated_paragraphs)

print("===== ENGLISH TRANSLATION =====")
print(translation)

===== ENGLISH TRANSLATION =====
Technology has revolutionized virtually every aspect of our lives, from how we communicate to how we perform our daily tasks, also changing the way we interact with the world around us. Its impact is evident in areas as diverse as education, health, transport, and trade, improving efficiency, accessibility and connectivity at levels never before imagined. However, despite the countless benefits it brings us, technology also brings with it complex and significant challenges that we cannot ignore.

One of the most important challenges is the digital divide, which exacerbates social and economic inequalities between those who have access to technologies and those who do not. This disparity limits the educational, work and social opportunities of many people in different parts of the world, leaving entire communities behind in technological development. Furthermore, the expansion of technology has raised serious concerns about the privacy and security of per

### 4.1 Translation Assessment

The generated English translation preserves the main meaning and structure of the Spanish source text across all three paragraphs.

Paragraph-level processing avoids sending the full document as a single long sequence and makes the translation workflow easier to inspect. The approach still depends on the capabilities and limitations of the pretrained translation model and does not include an external reference translation or quantitative translation metric.

## 5. Conclusions

This project demonstrates a compact NLP pipeline combining classical text processing with pretrained Transformer models.

Key observations:

- A deterministic **extractive summarization** strategy provides a simple and reproducible baseline.
- A pretrained multilingual BERT model can perform **sentiment analysis** without task-specific training, although long and mixed-polarity documents require careful interpretation.
- MarianMT provides effective **Spanish-to-English machine translation** using a pretrained sequence-to-sequence Transformer model.
- Pretrained models make it possible to build useful NLP pipelines with relatively little task-specific code.
- Model outputs must still be interpreted in the context of input length, task formulation and model limitations.

The project highlights both the practical value of pretrained NLP models and the importance of documenting methodological constraints when evaluating their outputs.